# Notebook 02-UNSW v2 — UNSW-NB15 Model Training

**Methodology changes vs v1:**

1. **M6 fix**: trained on the 80% train slice from Notebook 01-UNSW v2 (108,272 samples), not the full 135,341 post-Generic-drop training set. The remaining 20% (27,069 samples) is reserved for calibrator fitting in Notebook 03-UNSW v2.

2. **RF max_depth=20 cap**: addresses the v1 deep-tree problem where unconstrained RF grew to mean depth 46 (max 63), making TreeExplainer SHAP infeasible at standard 2000-sample size. With max_depth=20, TreeExplainer runs at full sample size and the architecture-specific subsample asymmetry (215 tree vs 509 DNN) goes away in downstream notebooks.

3. **Predictions on both test and calibration sets**: every model writes 4 prediction files (test_pred, test_proba, calib_pred, calib_proba). Notebook 03-UNSW v2 will use the calib_proba files to fit isotonic calibrators.

**Models trained (9 total):**

Canonical 6 (used by all downstream notebooks):
- `rf_binary_cw`, `xgb_binary_cw`, `dnn_binary_cw` — binary, class-weighted
- `rf_5class_smote`, `xgb_5class_smote`, `dnn_5class_smote` — 5-class, modified SMOTE

Ablation 3 (justifies SMOTE choice for 5-class):
- `rf_5class_cw`, `xgb_5class_cw`, `dnn_5class_cw` — 5-class, class-weighted

**Modified SMOTE design (preserved from v1):**
- Normal class kept as-is (no oversampling)
- All four attack classes oversampled to the size of the *largest attack class* (R2L)
- Avoids the over-synthesis problem of SMOTE-to-Normal-size (which would synthesise U2R 42×) while still balancing attack classes against each other

**Outputs written to `models/unsw_nb15_v2/`** — parallel to v1, leaves v1 intact.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
PROJECT_DIR = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(PROJECT_DIR)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull origin main
print(f'\n✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
From https://github.com/anasbiswas1/xids-research
 * branch            main       -> FETCH_HEAD
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
# GPU check
!nvidia-smi -L

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nPyTorch device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU. Switch to T4: Runtime → Change runtime type → T4 GPU.')

GPU 0: Tesla T4 (UUID: GPU-0924be97-ba12-863d-b186-2bcb8fdf6bbe)

PyTorch device: cuda
GPU: Tesla T4


## 2. Load v2 preprocessed data

In [3]:
import numpy as np
import pandas as pd
import json, joblib, time
from pathlib import Path
from collections import Counter

REPO       = Path(PROJECT_DIR)
PROC_DIR   = REPO / 'data/processed/unsw_nb15_v2'
MODELS_DIR = REPO / 'models/unsw_nb15_v2'
PREDS_DIR  = MODELS_DIR / 'predictions'
PROBS_DIR  = MODELS_DIR / 'probabilities'
TABLES_DIR = REPO / 'results/tables'
for d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

X_train = np.load(PROC_DIR / 'X_train.npy')
X_calib = np.load(PROC_DIR / 'X_calib.npy')
X_test  = np.load(PROC_DIR / 'X_test.npy')

y_train_binary = np.load(PROC_DIR / 'y_train_binary.npy')
y_calib_binary = np.load(PROC_DIR / 'y_calib_binary.npy')
y_test_binary  = np.load(PROC_DIR / 'y_test_binary.npy')

y_train_5class = np.load(PROC_DIR / 'y_train_5class.npy')
y_calib_5class = np.load(PROC_DIR / 'y_calib_5class.npy')
y_test_5class  = np.load(PROC_DIR / 'y_test_5class.npy')

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names = json.load(f)
with open(PROC_DIR / 'class_mappings.json') as f:
    class_mappings = json.load(f)

FIVE_CLASS_NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
N_FEATURES = X_train.shape[1]

print(f'Train: {X_train.shape}  (80% slice)')
print(f'Calib: {X_calib.shape}  (20% slice — fed to Notebook 03 v2)')
print(f'Test:  {X_test.shape}   (untouched official test)')
print(f'\nTrain 5-class distribution:')
for cid, name in enumerate(FIVE_CLASS_NAMES):
    n = int((y_train_5class == cid).sum())
    print(f'  {cid} {name:8s}: {n:>7,}')

Train: (108272, 194)  (80% slice)
Calib: (27069, 194)  (20% slice — fed to Notebook 03 v2)
Test:  (63461, 194)   (untouched official test)

Train 5-class distribution:
  0 Normal  :  44,800
  1 DoS     :   9,811
  2 Probe   :   9,993
  3 R2L     :  42,658
  4 U2R     :   1,010


## 3. Evaluation and save helpers

In [4]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix, roc_auc_score
)

ALL_METRICS = {}

def evaluate(name, y_true, y_pred, y_proba, target_type):
    metrics = {
        'model': name,
        'target_type': target_type,
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'recall_macro': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, y_pred)),
    }
    if target_type == 'binary':
        pos_proba = y_proba[:, 1] if y_proba.ndim == 2 else y_proba
        metrics['auc_roc'] = float(roc_auc_score(y_true, pos_proba))
        metrics['per_class_f1'] = {
            'Normal': float(f1_score(y_true, y_pred, pos_label=0)),
            'Attack': float(f1_score(y_true, y_pred, pos_label=1)),
        }
    else:
        per_class = f1_score(y_true, y_pred, average=None, labels=range(5), zero_division=0)
        metrics['per_class_f1'] = {n: float(per_class[i]) for i, n in enumerate(FIVE_CLASS_NAMES)}
    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred).tolist()
    return metrics

def save_artifacts(name, model, model_kind, predict_fn):
    """Save model, plus predictions on both test and calibration sets."""
    if model_kind == 'sklearn' or model_kind == 'xgboost':
        joblib.dump(model, MODELS_DIR / f'{name}.joblib')
    elif model_kind == 'torch':
        torch.save(model.state_dict(), MODELS_DIR / f'{name}.pt')

    test_pred, test_proba = predict_fn(X_test)
    calib_pred, calib_proba = predict_fn(X_calib)

    np.save(PREDS_DIR / f'{name}_test_pred.npy',   test_pred)
    np.save(PROBS_DIR / f'{name}_test_proba.npy',  test_proba)
    np.save(PREDS_DIR / f'{name}_calib_pred.npy',  calib_pred)
    np.save(PROBS_DIR / f'{name}_calib_proba.npy', calib_proba)

    return test_pred, test_proba

def checkpoint_metrics():
    with open(MODELS_DIR / 'metrics.json', 'w') as f:
        json.dump(ALL_METRICS, f, indent=2, default=str)

print('Helpers ready.')

Helpers ready.


## 4. RF binary, class-weighted (with max_depth=20 cap)

**max_depth=20** is new in v2. v1 used unconstrained depth which grew RF trees to mean 46 / max 63, making TreeExplainer infeasible. With max_depth=20, downstream SHAP can run at 2000 samples instead of 215.

In [5]:
from sklearn.ensemble import RandomForestClassifier

RF_MAX_DEPTH = 20  # NEW IN v2 — cap depth to enable TreeExplainer at full sample size

name = 'rf_binary_cw'
print(f'Training {name}...')
t0 = time.time()

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=RF_MAX_DEPTH,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)
clf.fit(X_train, y_train_binary)

def predict_fn(X):
    return clf.predict(X), clf.predict_proba(X)

test_pred, test_proba = save_artifacts(name, clf, 'sklearn', predict_fn)
metrics = evaluate(name, y_test_binary, test_pred, test_proba, 'binary')
metrics['train_time_sec'] = round(time.time() - t0, 1)
metrics['max_depth_actual'] = int(max(t.tree_.max_depth for t in clf.estimators_))
metrics['max_depth_mean'] = float(np.mean([t.tree_.max_depth for t in clf.estimators_]))
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}  AUC={metrics['auc_roc']:.4f}")
print(f"  Tree depths: mean={metrics['max_depth_mean']:.1f}, max={metrics['max_depth_actual']} (cap was {RF_MAX_DEPTH})")
print(f"  Time: {metrics['train_time_sec']}s")

Training rf_binary_cw...
  Acc=0.8550  F1m=0.8547  MCC=0.7344  AUC=0.9719
  Tree depths: mean=20.0, max=20 (cap was 20)
  Time: 37.6s


## 5. XGB binary, class-weighted

In [6]:
import xgboost as xgb

name = 'xgb_binary_cw'
print(f'Training {name}...')
t0 = time.time()

spw = float((y_train_binary == 0).sum()) / float((y_train_binary == 1).sum())
print(f'  scale_pos_weight = {spw:.3f}')

clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    tree_method='hist',
    device='cuda' if device.type == 'cuda' else 'cpu',
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train_binary)

def predict_fn(X):
    return clf.predict(X), clf.predict_proba(X)

test_pred, test_proba = save_artifacts(name, clf, 'xgboost', predict_fn)
metrics = evaluate(name, y_test_binary, test_pred, test_proba, 'binary')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}  AUC={metrics['auc_roc']:.4f}")
print(f"  Time: {metrics['train_time_sec']}s")

Training xgb_binary_cw...
  scale_pos_weight = 0.706


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [00:43:37] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  Acc=0.8553  F1m=0.8550  MCC=0.7329  AUC=0.9738
  Time: 7.0s


## 6. DNN architecture and training helper

In [7]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

class XIDSMLP(nn.Module):
    """256-128-64-32 MLP with BatchNorm, ReLU, dropout 0.3."""
    def __init__(self, n_features, n_classes, p_drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(256, 128),        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(128, 64),         nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(64, 32),          nn.BatchNorm1d(32),  nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(32, n_classes),
        )
    def forward(self, x):
        return self.net(x)

def train_dnn(X_tr, y_tr, n_classes, class_weights=None,
              epochs=40, batch_size=512, lr=1e-3, patience=5, verbose=True):
    """Train DNN with early stopping on a 90/10 stratified val split."""
    X_tr2, X_val, y_tr2, y_val = train_test_split(
        X_tr, y_tr, test_size=0.1, random_state=42, stratify=y_tr,
    )

    model = XIDSMLP(X_tr2.shape[1], n_classes).to(device)

    if class_weights is not None:
        cw_t = torch.tensor(class_weights, dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=cw_t)
    else:
        criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    Xtr = torch.tensor(X_tr2, dtype=torch.float32)
    ytr = torch.tensor(y_tr2, dtype=torch.long)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    Xv = torch.tensor(X_val, dtype=torch.float32).to(device)
    yv = torch.tensor(y_val, dtype=torch.long).to(device)

    best_val_loss = float('inf')
    best_state = None
    no_improve = 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_logits = model(Xv)
            val_loss = criterion(val_logits, yv).item()
            val_acc = (val_logits.argmax(1) == yv).float().mean().item()
        if verbose and (ep == 1 or ep % 5 == 0):
            print(f'  ep {ep:>3d}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose: print(f'  early stop at epoch {ep}')
                break

    model.load_state_dict(best_state)
    model.eval()
    return model

def dnn_predict(model):
    """Return predict_fn for use with save_artifacts."""
    def predict_fn(X):
        Xte = torch.tensor(X, dtype=torch.float32).to(device)
        model.eval()
        with torch.no_grad():
            logits = model(Xte)
            proba = F.softmax(logits, dim=1).cpu().numpy()
            pred = proba.argmax(axis=1)
        return pred, proba
    return predict_fn

print('DNN helpers ready.')

DNN helpers ready.


## 7. DNN binary, class-weighted

In [8]:
from sklearn.utils.class_weight import compute_class_weight

name = 'dnn_binary_cw'
print(f'Training {name}...')
t0 = time.time()

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train_binary)
print(f'  class weights: Normal={cw[0]:.3f}, Attack={cw[1]:.3f}')

model = train_dnn(X_train, y_train_binary, n_classes=2, class_weights=cw)
predict_fn = dnn_predict(model)
test_pred, test_proba = save_artifacts(name, model, 'torch', predict_fn)
metrics = evaluate(name, y_test_binary, test_pred, test_proba, 'binary')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}  AUC={metrics['auc_roc']:.4f}")
print(f"  Time: {metrics['train_time_sec']}s")

Training dnn_binary_cw...
  class weights: Normal=1.208, Attack=0.853
  ep   1  val_loss=0.1775  val_acc=0.9157
  ep   5  val_loss=0.1635  val_acc=0.9148
  ep  10  val_loss=0.1603  val_acc=0.9188
  ep  15  val_loss=0.1587  val_acc=0.9169
  ep  20  val_loss=0.1561  val_acc=0.9193
  ep  25  val_loss=0.1538  val_acc=0.9185
  ep  30  val_loss=0.1533  val_acc=0.9218
  ep  35  val_loss=0.1521  val_acc=0.9235
  ep  40  val_loss=0.1513  val_acc=0.9218
  Acc=0.8606  F1m=0.8599  MCC=0.7351  AUC=0.9677
  Time: 57.5s


## 8. Modified SMOTE on 5-class training data

Preserved from v1: oversample each attack class to the size of the largest attack class (R2L), not to Normal-class size. Avoids the over-synthesis problem of SMOTE-to-Normal-size.

In [9]:
from imblearn.over_sampling import SMOTE

print('Pre-SMOTE 5-class distribution:')
pre = Counter(y_train_5class)
for cid in range(5):
    print(f'  {cid} {FIVE_CLASS_NAMES[cid]:8s}: {pre[cid]:>7,}')

largest_attack = max(pre[c] for c in [1, 2, 3, 4])
sampling_strategy = {
    0: pre[0],
    1: max(pre[1], largest_attack),
    2: max(pre[2], largest_attack),
    3: max(pre[3], largest_attack),
    4: max(pre[4], largest_attack),
}

smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=5)
X_train_sm, y_train_5class_sm = smote.fit_resample(X_train, y_train_5class)
X_train_sm = X_train_sm.astype(np.float32)

print('\nPost-SMOTE distribution:')
post = Counter(y_train_5class_sm)
for cid in range(5):
    print(f'  {cid} {FIVE_CLASS_NAMES[cid]:8s}: {post[cid]:>7,}')
print(f'\nResampled shape: {X_train_sm.shape}')

Pre-SMOTE 5-class distribution:
  0 Normal  :  44,800
  1 DoS     :   9,811
  2 Probe   :   9,993
  3 R2L     :  42,658
  4 U2R     :   1,010

Post-SMOTE distribution:
  0 Normal  :  44,800
  1 DoS     :  42,658
  2 Probe   :  42,658
  3 R2L     :  42,658
  4 U2R     :  42,658

Resampled shape: (215432, 194)


## 9. RF 5-class SMOTE (with max_depth=20 cap)

In [10]:
# Check what variables exist
print('Variables in memory:')
for name in ['X_train', 'X_test', 'X_calib', 'y_train_5class',
             'X_train_sm', 'y_train_5class_sm',
             'X_train_binary', 'y_train_binary']:
    if name in dir():
        v = eval(name)
        try:
            print(f'  ✓ {name}: shape={v.shape}')
        except AttributeError:
            print(f'  ✓ {name}: {type(v).__name__}')
    else:
        print(f'  ✗ {name}: NOT IN MEMORY')

Variables in memory:
  ✓ X_train: shape=(108272, 194)
  ✓ X_test: shape=(63461, 194)
  ✓ X_calib: shape=(27069, 194)
  ✓ y_train_5class: shape=(108272,)
  ✓ X_train_sm: shape=(215432, 194)
  ✓ y_train_5class_sm: shape=(215432,)
  ✗ X_train_binary: NOT IN MEMORY
  ✓ y_train_binary: shape=(108272,)


In [11]:
name = 'rf_5class_smote'
print(f'Training {name}...')
t0 = time.time()

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=RF_MAX_DEPTH,
    n_jobs=-1,
    random_state=42,
)
clf.fit(X_train_sm, y_train_5class_sm)

def predict_fn(X):
    return clf.predict(X), clf.predict_proba(X)

test_pred, test_proba = save_artifacts(name, clf, 'sklearn', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
metrics['max_depth_actual'] = int(max(t.tree_.max_depth for t in clf.estimators_))
metrics['max_depth_mean'] = float(np.mean([t.tree_.max_depth for t in clf.estimators_]))
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")
print(f"  Per-class F1: {metrics['per_class_f1']}")
print(f"  Tree depths: mean={metrics['max_depth_mean']:.1f}, max={metrics['max_depth_actual']} (cap was {RF_MAX_DEPTH})")
print(f"  Time: {metrics['train_time_sec']}s")

Training rf_5class_smote...
  Acc=0.6872  F1m=0.5505  MCC=0.5297
  Per-class F1: {'Normal': 0.8288616179055263, 'DoS': 0.44092914828074264, 'Probe': 0.5990928810885426, 'R2L': 0.5730278025963105, 'U2R': 0.3104872006606111}
  Tree depths: mean=20.0, max=20 (cap was 20)
  Time: 87.8s


## 10. XGB 5-class SMOTE

In [12]:
name = 'xgb_5class_smote'
print(f'Training {name}...')
t0 = time.time()

clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    tree_method='hist',
    device='cuda' if device.type == 'cuda' else 'cpu',
    objective='multi:softprob',
    num_class=5,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train_sm, y_train_5class_sm)

def predict_fn(X):
    return clf.predict(X), clf.predict_proba(X)

test_pred, test_proba = save_artifacts(name, clf, 'xgboost', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")
print(f"  Per-class F1: {metrics['per_class_f1']}")
print(f"  Time: {metrics['train_time_sec']}s")

Training xgb_5class_smote...
  Acc=0.7152  F1m=0.5911  MCC=0.5608
  Per-class F1: {'Normal': 0.8463031748215405, 'DoS': 0.4436215515645251, 'Probe': 0.623978483500569, 'R2L': 0.6059959891351255, 'U2R': 0.4356314826113484}
  Time: 17.2s


## 11. DNN 5-class SMOTE

In [13]:
name = 'dnn_5class_smote'
print(f'Training {name}...')
t0 = time.time()

cw = compute_class_weight('balanced', classes=np.arange(5), y=y_train_5class_sm)
print(f'  class weights: {dict(zip(FIVE_CLASS_NAMES, cw.round(3)))}')

model = train_dnn(X_train_sm, y_train_5class_sm, n_classes=5, class_weights=cw)
predict_fn = dnn_predict(model)
test_pred, test_proba = save_artifacts(name, model, 'torch', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")
print(f"  Per-class F1: {metrics['per_class_f1']}")
print(f"  Time: {metrics['train_time_sec']}s")

Training dnn_5class_smote...
  class weights: {'Normal': np.float64(0.962), 'DoS': np.float64(1.01), 'Probe': np.float64(1.01), 'R2L': np.float64(1.01), 'U2R': np.float64(1.01)}
  ep   1  val_loss=0.7654  val_acc=0.6471
  ep   5  val_loss=0.5922  val_acc=0.7596
  ep  10  val_loss=0.5458  val_acc=0.7752
  ep  15  val_loss=0.5289  val_acc=0.7817
  ep  20  val_loss=0.5342  val_acc=0.7825
  ep  25  val_loss=0.5160  val_acc=0.7831
  ep  30  val_loss=0.5103  val_acc=0.7874
  ep  35  val_loss=0.5095  val_acc=0.7866
  ep  40  val_loss=0.5019  val_acc=0.7905
  Acc=0.6348  F1m=0.5020  MCC=0.4751
  Per-class F1: {'Normal': 0.7920511913566353, 'DoS': 0.44086773967809656, 'Probe': 0.5477185966187966, 'R2L': 0.5062157577105988, 'U2R': 0.22330935251798562}
  Time: 111.3s


## 12. Ablation: 5-class class-weighted variants

In [14]:
labels_5 = list(range(5))
cw_5 = compute_class_weight('balanced', classes=np.array(labels_5), y=y_train_5class)
cw_5_dict = {i: float(cw_5[i]) for i in labels_5}
sample_w_5 = np.array([cw_5[v] for v in y_train_5class])

print(f'5-class class weights:')
for i in range(5):
    print(f'  {FIVE_CLASS_NAMES[i]:8s}: {cw_5[i]:.4f}')

# RF 5-class CW
name = 'rf_5class_cw'
print(f'\nTraining {name}...')
t0 = time.time()
clf = RandomForestClassifier(n_estimators=200, max_depth=RF_MAX_DEPTH, class_weight=cw_5_dict, n_jobs=-1, random_state=42)
clf.fit(X_train, y_train_5class)
def predict_fn(X): return clf.predict(X), clf.predict_proba(X)
test_pred, test_proba = save_artifacts(name, clf, 'sklearn', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()
print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")

# XGB 5-class CW
name = 'xgb_5class_cw'
print(f'\nTraining {name}...')
t0 = time.time()
clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1, tree_method='hist',
    device='cuda' if device.type == 'cuda' else 'cpu',
    objective='multi:softprob', num_class=5, eval_metric='mlogloss',
    random_state=42, n_jobs=-1,
)
clf.fit(X_train, y_train_5class, sample_weight=sample_w_5)
def predict_fn(X): return clf.predict(X), clf.predict_proba(X)
test_pred, test_proba = save_artifacts(name, clf, 'xgboost', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()
print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")

# DNN 5-class CW
name = 'dnn_5class_cw'
print(f'\nTraining {name}...')
t0 = time.time()
model = train_dnn(X_train, y_train_5class, n_classes=5, class_weights=cw_5)
predict_fn = dnn_predict(model)
test_pred, test_proba = save_artifacts(name, model, 'torch', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()
print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")

5-class class weights:
  Normal  : 0.4834
  DoS     : 2.2072
  Probe   : 2.1670
  R2L     : 0.5076
  U2R     : 21.4400

Training rf_5class_cw...
  Acc=0.6961  F1m=0.5686  MCC=0.5376

Training xgb_5class_cw...
  Acc=0.7012  F1m=0.5775  MCC=0.5440

Training dnn_5class_cw...
  ep   1  val_loss=0.8877  val_acc=0.6037
  ep   5  val_loss=0.7760  val_acc=0.6248
  ep  10  val_loss=0.6948  val_acc=0.6725
  ep  15  val_loss=0.6717  val_acc=0.6772
  ep  20  val_loss=0.6465  val_acc=0.6909
  ep  25  val_loss=0.6288  val_acc=0.6933
  ep  30  val_loss=0.6319  val_acc=0.6765
  ep  35  val_loss=0.6223  val_acc=0.6987
  early stop at epoch 38
  Acc=0.6017  F1m=0.4774  MCC=0.4453


## 13. Summary table

In [15]:
rows = []
for name, m in ALL_METRICS.items():
    row = {
        'Model': name,
        'Target': m['target_type'],
        'Accuracy': round(m['accuracy'], 4),
        'F1-macro': round(m['f1_macro'], 4),
        'F1-weighted': round(m['f1_weighted'], 4),
        'MCC': round(m['mcc'], 4),
        'AUC-ROC': round(m['auc_roc'], 4) if m['target_type'] == 'binary' else None,
        'R2L F1': round(m['per_class_f1']['R2L'], 4) if 'R2L' in m['per_class_f1'] else None,
        'U2R F1': round(m['per_class_f1']['U2R'], 4) if 'U2R' in m['per_class_f1'] else None,
        'Time (s)': m['train_time_sec'],
    }
    rows.append(row)
df = pd.DataFrame(rows)
print('ALL 9 MODELS — UNSW-NB15 v2')
print('=' * 110)
print(df.to_string(index=False))
print('=' * 110)
df.to_csv(TABLES_DIR / 'unsw_v2_model_comparison.csv', index=False)
print(f'\nSaved: {TABLES_DIR / "unsw_v2_model_comparison.csv"}')

ALL 9 MODELS — UNSW-NB15 v2
           Model     Target  Accuracy  F1-macro  F1-weighted    MCC  AUC-ROC  R2L F1  U2R F1  Time (s)
    rf_binary_cw     binary    0.8550    0.8547       0.8558 0.7344   0.9719     NaN     NaN      37.6
   xgb_binary_cw     binary    0.8553    0.8550       0.8562 0.7329   0.9738     NaN     NaN       7.0
   dnn_binary_cw     binary    0.8606    0.8599       0.8615 0.7351   0.9677     NaN     NaN      57.5
 rf_5class_smote multiclass    0.6872    0.5505       0.7136 0.5297      NaN  0.5730  0.3105      87.8
xgb_5class_smote multiclass    0.7152    0.5911       0.7357 0.5608      NaN  0.6060  0.4356      17.2
dnn_5class_smote multiclass    0.6348    0.5020       0.6695 0.4751      NaN  0.5062  0.2233     111.3
    rf_5class_cw multiclass    0.6961    0.5686       0.7192 0.5376      NaN  0.5891  0.3789      40.2
   xgb_5class_cw multiclass    0.7012    0.5775       0.7236 0.5440      NaN  0.5874  0.4187      12.0
   dnn_5class_cw multiclass    0.6017    0.47

## 14. Imbalance-strategy ablation table

In [16]:
ablation_rows = []
for arch in ['rf', 'xgb', 'dnn']:
    for strat in ['cw', 'smote']:
        name = f'{arch}_5class_{strat}'
        m = ALL_METRICS[name]
        ablation_rows.append({
            'Architecture': arch.upper(),
            'Strategy': 'class-weighted' if strat == 'cw' else 'modified-SMOTE',
            'Accuracy': round(m['accuracy'], 4),
            'Macro F1': round(m['f1_macro'], 4),
            'MCC': round(m['mcc'], 4),
            'R2L F1': round(m['per_class_f1']['R2L'], 4),
            'U2R F1': round(m['per_class_f1']['U2R'], 4),
        })
df = pd.DataFrame(ablation_rows)
print('5-CLASS IMBALANCE STRATEGY ABLATION — UNSW-NB15 v2')
print('=' * 100)
print(df.to_string(index=False))
print('=' * 100)
df.to_csv(TABLES_DIR / 'unsw_v2_imbalance_ablation.csv', index=False)

5-CLASS IMBALANCE STRATEGY ABLATION — UNSW-NB15 v2
Architecture       Strategy  Accuracy  Macro F1    MCC  R2L F1  U2R F1
          RF class-weighted    0.6961    0.5686 0.5376  0.5891  0.3789
          RF modified-SMOTE    0.6872    0.5505 0.5297  0.5730  0.3105
         XGB class-weighted    0.7012    0.5775 0.5440  0.5874  0.4187
         XGB modified-SMOTE    0.7152    0.5911 0.5608  0.6060  0.4356
         DNN class-weighted    0.6017    0.4774 0.4453  0.4799  0.1558
         DNN modified-SMOTE    0.6348    0.5020 0.4751  0.5062  0.2233


## 15. Commit and push

In [17]:
os.chdir(PROJECT_DIR)
!git add notebooks/02_unsw_train_models_v2.ipynb
!git add results/tables/unsw_v2_model_comparison.csv
!git add results/tables/unsw_v2_imbalance_ablation.csv
!git status --short
!git commit -m 'Notebook 02-UNSW v2: train 9 models on 80% slice with RF max_depth=20 cap (kills depth-46 issue)'
!git push origin main

Refresh index: 100% (230/230), done.
 M notebooks/01_cic_data_exploration_v2.ipynb
 M notebooks/01_data_exploration_v2.ipynb
 M notebooks/01_unsw_data_exploration_v2.ipynb
 M notebooks/02_train_models_v2.ipynb
AM notebooks/02_unsw_train_models_v2.ipynb
A  results/tables/unsw_v2_imbalance_ablation.csv
A  results/tables/unsw_v2_model_comparison.csv
?? calibrators/
?? models/
?? notebooks/02b_unsw_dnn_diagnostic.ipynb
?? results/figures/unsw_dnn_5class_diagnostic.png
?? shap_values/unsw_nb15/
[main 8fb48a4] Notebook 02-UNSW v2: train 9 models on 80% slice with RF max_depth=20 cap (kills depth-46 issue)
 3 files changed, 18 insertions(+)
 create mode 100644 notebooks/02_unsw_train_models_v2.ipynb
 create mode 100644 results/tables/unsw_v2_imbalance_ablation.csv
 create mode 100644 results/tables/unsw_v2_model_comparison.csv
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100